In [ ]:
# ============================================================
# PRE-ABLATION SETUP CELL
# Creates:
#   train_loader, val_loader, test_loader
#   CSI_CHANNELS, NUM_CLASSES
#   LABEL_MAPPING, INV_LABEL_MAPPING
#
# Run this BEFORE the DRFT-LSTM ablation cell.
# ============================================================

import os
import re
import gc
import glob
import json
import h5py
import random
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIG
# ============================================================

TASK_NAME = "HumanActivityRecognition"

SEED = 42

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 0

TARGET_SUBCARRIERS = 56
TARGET_TIME_LEN = 500

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("medium")

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


set_seed(SEED)
cleanup()

# ============================================================
# AUTO-DETECT CSI-BENCH ROOT
# ============================================================

def find_task_root(task_name="HumanActivityRecognition"):
    patterns = [
        f"/kaggle/input/datasets/guozhenjennzhu/csi-bench/Multitask/{task_name}",
        f"/kaggle/input/**/Multitask/{task_name}",
        f"/kaggle/input/**/{task_name}",
    ]

    candidates = []

    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            if os.path.isdir(path):
                candidates.append(path)

    candidates = list(dict.fromkeys(candidates))

    print("\nCandidate task roots:")
    for path in candidates:
        print(" -", path)

    for path in candidates:
        split_file = os.path.join(path, "splits", "train_id.json")
        metadata_file = os.path.join(path, "metadata", "sample_metadata.csv")

        if os.path.exists(split_file) and os.path.exists(metadata_file):
            print("\n✅ Using ROOT:")
            print(path)
            return path

    raise FileNotFoundError(
        "Could not find HumanActivityRecognition root with splits/train_id.json "
        "and metadata/sample_metadata.csv"
    )


ROOT = find_task_root(TASK_NAME)

METADATA_DIR = os.path.join(ROOT, "metadata")
SPLITS_DIR = os.path.join(ROOT, "splits")
MULTITASK_DIR = os.path.dirname(ROOT)
CSI_BENCH_DIR = os.path.dirname(MULTITASK_DIR)

print("\n========== ROOT VERIFICATION ==========")
print("ROOT:", ROOT)
print("Has metadata:", os.path.exists(METADATA_DIR))
print("Has splits:", os.path.exists(SPLITS_DIR))
print("Train split:", os.path.exists(os.path.join(SPLITS_DIR, "train_id.json")))
print("Val split:", os.path.exists(os.path.join(SPLITS_DIR, "val_id.json")))
print("Test split:", os.path.exists(os.path.join(SPLITS_DIR, "test_id.json")))
print("Metadata CSV:", os.path.exists(os.path.join(METADATA_DIR, "sample_metadata.csv")))
print("sub_Human_h5:", os.path.exists(os.path.join(MULTITASK_DIR, "sub_Human_h5")))

# ============================================================
# SPLIT + LABEL SETUP
# ============================================================

def load_split_ids(root_dir, split):
    split_file = os.path.join(root_dir, "splits", f"{split}_id.json")

    if not os.path.exists(split_file):
        raise FileNotFoundError(f"Missing split file: {split_file}")

    with open(split_file, "r") as f:
        return set(map(str, json.load(f)))


metadata_path = os.path.join(ROOT, "metadata", "sample_metadata.csv")
metadata = pd.read_csv(metadata_path)

metadata["id"] = metadata["id"].astype(str)
metadata["file_path"] = metadata["file_path"].astype(str)

train_ids = load_split_ids(ROOT, "train")
val_ids = load_split_ids(ROOT, "val")
test_ids = load_split_ids(ROOT, "test")

print("\n========== SPLIT CHECK ==========")
print("Train IDs:", len(train_ids))
print("Val IDs  :", len(val_ids))
print("Test IDs :", len(test_ids))
print("Train-Val overlap :", len(train_ids & val_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Val-Test overlap  :", len(val_ids & test_ids))

train_meta = metadata[metadata["id"].isin(train_ids)].copy()
train_labels = sorted(train_meta["label"].unique())

LABEL_MAPPING = {label: idx for idx, label in enumerate(train_labels)}
INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}
NUM_CLASSES = len(LABEL_MAPPING)

print("\nLabel mapping:")
print(LABEL_MAPPING)

# ============================================================
# DATASET
# ============================================================

class CSIBenchDataset(Dataset):
    def __init__(
        self,
        root_dir,
        split="train",
        normalize=True,
        target_subcarriers=56,
        target_time_len=500,
        label_mapping=None,
    ):
        self.root_dir = root_dir
        self.split = split
        self.normalize = normalize
        self.target_subcarriers = target_subcarriers
        self.target_time_len = target_time_len
        self.label_mapping = label_mapping

        self.metadata_dir = os.path.join(root_dir, "metadata")
        self.splits_dir = os.path.join(root_dir, "splits")
        self.multitask_dir = os.path.dirname(root_dir)
        self.csi_bench_dir = os.path.dirname(self.multitask_dir)

        split_file = os.path.join(self.splits_dir, f"{split}_id.json")
        metadata_file = os.path.join(self.metadata_dir, "sample_metadata.csv")

        if not os.path.exists(split_file):
            raise FileNotFoundError(f"Split file not found: {split_file}")

        if not os.path.exists(metadata_file):
            raise FileNotFoundError(f"Metadata file not found: {metadata_file}")

        with open(split_file, "r") as f:
            self.sample_ids = list(map(str, json.load(f)))

        self.metadata = pd.read_csv(metadata_file)
        self.metadata["id"] = self.metadata["id"].astype(str)
        self.metadata["file_path"] = self.metadata["file_path"].astype(str)

        self.meta_dict = {
            str(row["id"]): row
            for _, row in self.metadata.iterrows()
        }

        print(f"Loaded {len(self.sample_ids)} samples for split: {split}")

    def __len__(self):
        return len(self.sample_ids)

    def resolve_file_path(self, raw_path):
        raw = str(raw_path).replace("\\", "/").strip()

        # Important fix for paths like:
        # ../../sub_Human_h5/user_U01/...
        while raw.startswith("../"):
            raw = raw[3:]

        if raw.startswith("./"):
            raw = raw[2:]

        candidates = []

        if os.path.isabs(raw):
            candidates.append(os.path.normpath(raw))

        candidates.extend([
            os.path.normpath(os.path.join(self.metadata_dir, raw)),
            os.path.normpath(os.path.join(self.root_dir, raw)),
            os.path.normpath(os.path.join(self.multitask_dir, raw)),
            os.path.normpath(os.path.join(self.csi_bench_dir, raw)),
            os.path.normpath(os.path.join("/kaggle/input", raw)),
        ])

        if "sub_Human_h5/" in raw:
            suffix = raw.split("sub_Human_h5/", 1)[1]
            candidates.append(
                os.path.normpath(
                    os.path.join(self.multitask_dir, "sub_Human_h5", suffix)
                )
            )
            candidates.append(
                os.path.normpath(
                    os.path.join(self.csi_bench_dir, "Multitask", "sub_Human_h5", suffix)
                )
            )

        if "sub_Human_mat/" in raw:
            suffix = raw.split("sub_Human_mat/", 1)[1]
            candidates.append(
                os.path.normpath(
                    os.path.join(self.multitask_dir, "sub_Human_mat", suffix)
                )
            )
            candidates.append(
                os.path.normpath(
                    os.path.join(self.csi_bench_dir, "Multitask", "sub_Human_mat", suffix)
                )
            )

        for path in candidates:
            if os.path.exists(path):
                return path

        base = os.path.basename(raw)
        matches = []

        for search_base in [self.multitask_dir, self.csi_bench_dir, "/kaggle/input"]:
            pattern = os.path.join(search_base, "**", base)
            matches.extend(glob.glob(pattern, recursive=True))

        matches = sorted(list(set(matches)))

        if len(matches) == 1:
            return matches[0]

        if len(matches) > 1:
            raise RuntimeError(
                "Ambiguous file resolution. Multiple files share the same basename.\n"
                f"metadata file_path: {raw_path}\n"
                f"cleaned path: {raw}\n"
                f"basename: {base}\n"
                "Matches:\n" + "\n".join(matches[:20])
            )

        raise FileNotFoundError(
            "Could not resolve CSI file path.\n"
            f"metadata file_path: {raw_path}\n"
            f"cleaned path: {raw}\n"
            f"basename searched: {base}"
        )

    def load_h5(self, path):
        with h5py.File(path, "r") as f:
            keys = list(f.keys())

            for key in ["csi", "data", "CSI", "amplitude"]:
                if key in keys:
                    return f[key][:]

            return f[keys[0]][:]

    def to_ckt(self, data):
        data = np.array(data)

        if np.iscomplexobj(data):
            data = np.abs(data)

        data = data.astype(np.float32)

        if data.ndim == 2:
            a, b = data.shape

            if a <= b:
                return data[np.newaxis, :, :]
            else:
                return data.T[np.newaxis, :, :]

        if data.ndim == 3:
            s0, s1, s2 = data.shape

            if s0 <= 8 and s1 <= self.target_subcarriers * 2:
                return data

            if s2 <= 8 and s0 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 0, 1))

            if s2 <= 8 and s1 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 1, 0))

            if s2 <= 16:
                return np.transpose(data, (2, 0, 1))

        raise ValueError(f"Unexpected CSI shape: {data.shape}")

    def standardize_subcarriers(self, x):
        C, K, T = x.shape

        if K > self.target_subcarriers:
            x = x[:, :self.target_subcarriers, :]
        elif K < self.target_subcarriers:
            pad = np.zeros((C, self.target_subcarriers - K, T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=1)

        return x

    def standardize_time(self, x):
        C, K, T = x.shape

        if T > self.target_time_len:
            x = x[:, :, :self.target_time_len]
        elif T < self.target_time_len:
            pad = np.zeros((C, K, self.target_time_len - T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=2)

        return x

    def __getitem__(self, idx):
        sample_id = str(self.sample_ids[idx])

        if sample_id not in self.meta_dict:
            raise KeyError(f"Sample ID not found in metadata: {sample_id}")

        meta = self.meta_dict[sample_id]

        file_path = self.resolve_file_path(meta["file_path"])
        csi = self.load_h5(file_path)

        x = self.to_ckt(csi)
        x = self.standardize_subcarriers(x)
        x = self.standardize_time(x)

        if self.normalize:
            x = (x - x.mean()) / (x.std() + 1e-6)

        label_name = meta["label"]

        if label_name not in self.label_mapping:
            raise KeyError(f"Label not in mapping: {label_name}")

        y = self.label_mapping[label_name]

        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = CSIBenchDataset(
    ROOT,
    split="train",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING,
)

val_dataset = CSIBenchDataset(
    ROOT,
    split="val",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING,
)

test_dataset = CSIBenchDataset(
    ROOT,
    split="test",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING,
)

# Force one sample load to verify path resolution and shape.
sample_x, sample_y = train_dataset[0]

CSI_CHANNELS = sample_x.shape[0]

print("\n========== DATA INFO ==========")
print("Sample shape:", sample_x.shape)
print("CSI channels:", CSI_CHANNELS)
print("Num classes:", NUM_CLASSES)
print("Example label:", sample_y)

# ============================================================
# CREATE LOADERS
# ============================================================

def make_loaders(seed=42):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        generator=generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
    )

    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = make_loaders(SEED)

print("\n========== READY FOR ABLATION CELL ==========")
print("Available variables:")
print("train_loader:", type(train_loader))
print("val_loader  :", type(val_loader))
print("test_loader :", type(test_loader))
print("CSI_CHANNELS:", CSI_CHANNELS)
print("NUM_CLASSES :", NUM_CLASSES)
print("LABEL_MAPPING:", LABEL_MAPPING)

cleanup()

In [ ]:
# ============================================================
# DRFT-LSTM Component Ablation: D1 to D4 + Robustness
#
# D1 = Fine temporal branch only
# D2 = Coarse temporal branch only
# D3 = Fine + Coarse dual-resolution concat fusion
# D4 = Full DRFT-LSTM: Fine + Coarse + Frequency + adaptive gate
#
# Run after dataset/loaders are already created.
# ============================================================

import os
import gc
import json
import time
import copy
import random
import shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from IPython.display import FileLink, display

# ============================================================
# CONFIG
# ============================================================

ABLATION_OUT_DIR = "/kaggle/working/drft_lstm_ablation_d1_d4"
os.makedirs(ABLATION_OUT_DIR, exist_ok=True)

SEED = 42

ABLATION_EPOCHS = 35
ABLATION_PATIENCE = 8
MIN_DELTA = 1e-4

BATCH_SIZE = globals().get("BATCH_SIZE", 8)
EVAL_BATCH_SIZE = globals().get("EVAL_BATCH_SIZE", 32)
ACCUM_STEPS = globals().get("ACCUM_STEPS", 4)

LR = globals().get("LR", 8e-4)
WEIGHT_DECAY = globals().get("WEIGHT_DECAY", 1e-4)
MAX_GRAD_NORM = globals().get("MAX_GRAD_NORM", 1.0)
LABEL_SMOOTHING = globals().get("LABEL_SMOOTHING", 0.03)

TARGET_SUBCARRIERS = globals().get("TARGET_SUBCARRIERS", 56)
TARGET_TIME_LEN = globals().get("TARGET_TIME_LEN", 500)

DROPOUT = globals().get("DROPOUT", 0.1)

FINE_HIDDEN = globals().get("FINE_HIDDEN", 96)
COARSE_HIDDEN = globals().get("COARSE_HIDDEN", 64)
FREQ_HIDDEN = globals().get("FREQ_HIDDEN", 64)
FUSION_DIM = globals().get("FUSION_DIM", 96)

NUM_FINE_LAYERS = globals().get("NUM_FINE_LAYERS", 2)
NUM_COARSE_LAYERS = globals().get("NUM_COARSE_LAYERS", 1)
NUM_FREQ_LAYERS = globals().get("NUM_FREQ_LAYERS", 1)

FREQ_BINS = globals().get("FREQ_BINS", 8)
COARSE_DOWNSAMPLE = globals().get("COARSE_DOWNSAMPLE", 4)

ROBUST_TRAINING = globals().get("ROBUST_TRAINING", True)
AUG_PROB = globals().get("AUG_PROB", 0.35)
AUG_RANDOM_SC_MAX = globals().get("AUG_RANDOM_SC_MAX", 0.25)
AUG_CONTIG_SC_MAX = globals().get("AUG_CONTIG_SC_MAX", 0.20)
AUG_NOISE_STD_MAX = globals().get("AUG_NOISE_STD_MAX", 0.05)

ROBUSTNESS_TRIALS = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUN_VARIANTS = [
    "D1_fine_only",
    "D2_coarse_only",
    "D3_dual_concat",
    "D4_full_gated",
]

required_vars = [
    "train_loader",
    "val_loader",
    "test_loader",
    "CSI_CHANNELS",
    "NUM_CLASSES",
]

missing = [v for v in required_vars if v not in globals()]
if missing:
    raise NameError(
        "Missing required variables from previous cells: "
        + ", ".join(missing)
        + "\nRun the dataset/loading part of your DRFT-LSTM notebook first."
    )

print("Using device:", DEVICE)
print("Running variants:", RUN_VARIANTS)
print("Output dir:", ABLATION_OUT_DIR)

# ============================================================
# REPRO / CLEANUP
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

set_seed(SEED)
cleanup()

# ============================================================
# ROBUST TRAINING AUGMENTATION
# ============================================================

def random_subcarrier_mask_batch(x, drop_prob):
    B, C, K, T = x.shape
    mask = (torch.rand(B, 1, K, 1, device=x.device) > drop_prob).float()
    return x * mask

def contiguous_subcarrier_mask_batch(x, drop_ratio):
    B, C, K, T = x.shape
    width = max(1, int(K * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, K - width + 1, (1,), device=x.device).item()
        out[b, :, start:start + width, :] = 0.0

    return out

def apply_train_augmentation(x):
    if not ROBUST_TRAINING:
        return x

    if torch.rand(1).item() > AUG_PROB:
        return x

    aug_choice = torch.rand(1).item()

    if aug_choice < 0.40:
        drop_prob = float(np.random.uniform(0.05, AUG_RANDOM_SC_MAX))
        x = random_subcarrier_mask_batch(x, drop_prob)

    elif aug_choice < 0.75:
        drop_ratio = float(np.random.uniform(0.05, AUG_CONTIG_SC_MAX))
        x = contiguous_subcarrier_mask_batch(x, drop_ratio)

    else:
        std = float(np.random.uniform(0.01, AUG_NOISE_STD_MAX))
        x = x + torch.randn_like(x) * std

    return x

# ============================================================
# MODEL COMPONENTS
# ============================================================

class AttentionPool1D(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.score = nn.Linear(dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.score(x), dim=1)
        return torch.sum(weights * x, dim=1)


class FrequencyFeatureEncoder(nn.Module):
    def __init__(self, csi_channels, freq_hidden=64, freq_bins=8, dropout=0.1):
        super().__init__()

        self.freq_bins = freq_bins

        self.freq_conv = nn.Sequential(
            nn.Conv1d(csi_channels, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(16),
            nn.GELU(),

            nn.Conv1d(16, 16, kernel_size=5, padding=2, groups=16, bias=False),
            nn.Conv1d(16, 32, kernel_size=1, bias=False),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.AdaptiveAvgPool1d(freq_bins),
        )

        self.proj = nn.Sequential(
            nn.LayerNorm(32 * freq_bins),
            nn.Linear(32 * freq_bins, freq_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        B, C, K, T = x.shape

        z = x.permute(0, 3, 1, 2).contiguous()
        z = z.reshape(B * T, C, K)

        z = self.freq_conv(z)
        z = z.reshape(B * T, -1)
        z = self.proj(z)

        z = z.reshape(B, T, -1)

        return z


class DRFTAblationLSTM(nn.Module):
    def __init__(
        self,
        variant,
        num_classes,
        csi_channels,
        num_subcarriers,
        fine_hidden=96,
        coarse_hidden=64,
        freq_hidden=64,
        fusion_dim=96,
        dropout=0.1,
    ):
        super().__init__()

        self.variant = variant
        self.input_dim = csi_channels * num_subcarriers
        self.coarse_downsample = COARSE_DOWNSAMPLE

        assert variant in [
            "D1_fine_only",
            "D2_coarse_only",
            "D3_dual_concat",
            "D4_full_gated",
        ]

        # Fine branch
        if variant in ["D1_fine_only", "D3_dual_concat", "D4_full_gated"]:
            self.fine_input_proj = nn.Sequential(
                nn.LayerNorm(self.input_dim),
                nn.Linear(self.input_dim, fine_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            )

            self.fine_lstm = nn.LSTM(
                input_size=fine_hidden,
                hidden_size=fine_hidden,
                num_layers=NUM_FINE_LAYERS,
                batch_first=True,
                dropout=dropout if NUM_FINE_LAYERS > 1 else 0.0,
                bidirectional=False,
            )

            self.fine_pool = AttentionPool1D(fine_hidden)
            self.fine_to_fusion = nn.Linear(fine_hidden, fusion_dim)

        # Coarse branch
        if variant in ["D2_coarse_only", "D3_dual_concat", "D4_full_gated"]:
            self.coarse_input_proj = nn.Sequential(
                nn.LayerNorm(self.input_dim),
                nn.Linear(self.input_dim, coarse_hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            )

            self.coarse_lstm = nn.LSTM(
                input_size=coarse_hidden,
                hidden_size=coarse_hidden,
                num_layers=NUM_COARSE_LAYERS,
                batch_first=True,
                dropout=dropout if NUM_COARSE_LAYERS > 1 else 0.0,
                bidirectional=False,
            )

            self.coarse_pool = AttentionPool1D(coarse_hidden)
            self.coarse_to_fusion = nn.Linear(coarse_hidden, fusion_dim)

        # Frequency branch only in full model
        if variant == "D4_full_gated":
            self.freq_encoder = FrequencyFeatureEncoder(
                csi_channels=csi_channels,
                freq_hidden=freq_hidden,
                freq_bins=FREQ_BINS,
                dropout=dropout,
            )

            self.freq_lstm = nn.LSTM(
                input_size=freq_hidden,
                hidden_size=freq_hidden,
                num_layers=NUM_FREQ_LAYERS,
                batch_first=True,
                dropout=dropout if NUM_FREQ_LAYERS > 1 else 0.0,
                bidirectional=False,
            )

            self.freq_pool = AttentionPool1D(freq_hidden)
            self.freq_to_fusion = nn.Linear(freq_hidden, fusion_dim)

            self.branch_gate = nn.Sequential(
                nn.LayerNorm(fusion_dim * 3),
                nn.Linear(fusion_dim * 3, fusion_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(fusion_dim, 3),
            )

        # Fusion strategy
        if variant == "D3_dual_concat":
            self.concat_fusion = nn.Sequential(
                nn.LayerNorm(fusion_dim * 2),
                nn.Linear(fusion_dim * 2, fusion_dim),
                nn.GELU(),
                nn.Dropout(dropout),
            )

        self.head = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim, num_classes),
        )

    def encode_fine(self, x):
        B, C, K, T = x.shape
        x_flat = x.reshape(B, C * K, T).transpose(1, 2)
        z = self.fine_input_proj(x_flat)
        z, _ = self.fine_lstm(z)
        z = self.fine_pool(z)
        z = self.fine_to_fusion(z)
        return z

    def encode_coarse(self, x):
        B, C, K, T = x.shape
        x_coarse = x.reshape(B, C * K, T)
        x_coarse = F.avg_pool1d(
            x_coarse,
            kernel_size=self.coarse_downsample,
            stride=self.coarse_downsample,
        )
        x_coarse = x_coarse.transpose(1, 2)

        z = self.coarse_input_proj(x_coarse)
        z, _ = self.coarse_lstm(z)
        z = self.coarse_pool(z)
        z = self.coarse_to_fusion(z)
        return z

    def encode_freq(self, x):
        z = self.freq_encoder(x)
        z, _ = self.freq_lstm(z)
        z = self.freq_pool(z)
        z = self.freq_to_fusion(z)
        return z

    def forward(self, x):
        if self.variant == "D1_fine_only":
            fused = self.encode_fine(x)

        elif self.variant == "D2_coarse_only":
            fused = self.encode_coarse(x)

        elif self.variant == "D3_dual_concat":
            fine_f = self.encode_fine(x)
            coarse_f = self.encode_coarse(x)

            fused = self.concat_fusion(
                torch.cat([fine_f, coarse_f], dim=-1)
            )

        elif self.variant == "D4_full_gated":
            fine_f = self.encode_fine(x)
            coarse_f = self.encode_coarse(x)
            freq_f = self.encode_freq(x)

            concat = torch.cat([fine_f, coarse_f, freq_f], dim=-1)
            gate_logits = self.branch_gate(concat)
            gate_weights = torch.softmax(gate_logits, dim=-1)

            fused = (
                gate_weights[:, 0:1] * fine_f
                + gate_weights[:, 1:2] * coarse_f
                + gate_weights[:, 2:3] * freq_f
            )

        logits = self.head(fused)

        return logits

# ============================================================
# TRAIN / EVAL FUNCTIONS
# ============================================================

def train_one_epoch_model(model_obj, loader, criterion, optimizer, scaler, epoch, variant_name):
    model_obj.train()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc=f"{variant_name} Epoch {epoch} Train", leave=False)

    for step, (x, y) in enumerate(progress):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        x = apply_train_augmentation(x)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model_obj(x)
            loss = criterion(logits, y)
            loss = loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model_obj.parameters(), MAX_GRAD_NORM)

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * ACCUM_STEPS

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        progress.set_postfix(loss=f"{running_loss / (step + 1):.4f}")

        del x, y, logits, loss, preds

    return {
        "loss": running_loss / len(loader),
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
    }


@torch.no_grad()
def evaluate_model_loader(model_obj, loader, criterion=None, split_name="Val"):
    model_obj.eval()

    running_loss = 0.0
    preds_all = []
    labels_all = []

    progress = tqdm(loader, desc=split_name, leave=False)

    for x, y in progress:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model_obj(x)

            if criterion is not None:
                loss = criterion(logits, y)
                running_loss += loss.item()

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, preds

    out = {
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
        "labels": labels_all,
        "preds": preds_all,
    }

    if criterion is not None:
        out["loss"] = running_loss / len(loader)
    else:
        out["loss"] = None

    return out


def model_size_mb(model_obj, out_dir):
    temp_path = os.path.join(out_dir, "temp_model_size.pth")
    torch.save(model_obj.state_dict(), temp_path)
    size_mb = os.path.getsize(temp_path) / (1024 ** 2)
    os.remove(temp_path)
    return size_mb


@torch.no_grad()
def profile_latency(model_obj, device, input_shape, warmup=20, runs=50):
    model_obj.eval()
    dummy = torch.randn(*input_shape).to(device)

    for _ in range(warmup):
        _ = model_obj(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    for _ in range(runs):
        _ = model_obj(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    latency_ms = ((end - start) / runs) * 1000

    peak_mem_mb = None

    if device.type == "cuda":
        peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return latency_ms, peak_mem_mb

# ============================================================
# ROBUSTNESS PERTURBATIONS
# ============================================================

def perturb_clean(x):
    return x

def perturb_gaussian_noise(x, std=0.2):
    return x + torch.randn_like(x) * std

def perturb_random_subcarrier_mask(x, drop_prob=0.3):
    B, C, K, T = x.shape
    mask = (torch.rand(B, 1, K, 1, device=x.device) > drop_prob).float()
    return x * mask

def perturb_contiguous_subcarrier_mask(x, drop_ratio=0.3):
    B, C, K, T = x.shape
    width = max(1, int(K * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, K - width + 1, (1,), device=x.device).item()
        out[b, :, start:start + width, :] = 0.0

    return out

def perturb_temporal_mask(x, drop_ratio=0.3):
    B, C, K, T = x.shape
    width = max(1, int(T * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, T - width + 1, (1,), device=x.device).item()
        out[b, :, :, start:start + width] = 0.0

    return out

def perturb_time_shift(x, shift=50):
    return torch.roll(x, shifts=shift, dims=-1)

def perturb_crop_resize(x, keep_ratio=0.75):
    B, C, K, T = x.shape
    keep_len = max(8, int(T * keep_ratio))
    out_list = []

    for b in range(B):
        start = torch.randint(0, T - keep_len + 1, (1,), device=x.device).item()
        crop = x[b:b + 1, :, :, start:start + keep_len]
        crop = crop.reshape(1, C * K, keep_len)

        resized = F.interpolate(
            crop,
            size=T,
            mode="linear",
            align_corners=False,
        )

        resized = resized.reshape(1, C, K, T)
        out_list.append(resized)

    return torch.cat(out_list, dim=0)

def perturb_combined_harsh(x):
    x = perturb_gaussian_noise(x, std=0.20)
    x = perturb_random_subcarrier_mask(x, drop_prob=0.40)
    x = perturb_temporal_mask(x, drop_ratio=0.20)
    return x


ROBUSTNESS_CONDITIONS = [
    {"name": "clean", "fn": perturb_clean, "kwargs": {}, "trials": 1},
    {"name": "gaussian_noise_0.20", "fn": perturb_gaussian_noise, "kwargs": {"std": 0.20}, "trials": ROBUSTNESS_TRIALS},
    {"name": "random_subcarrier_mask_10", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.10}, "trials": ROBUSTNESS_TRIALS},
    {"name": "random_subcarrier_mask_30", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "random_subcarrier_mask_50", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.50}, "trials": ROBUSTNESS_TRIALS},
    {"name": "contiguous_subcarrier_mask_10", "fn": perturb_contiguous_subcarrier_mask, "kwargs": {"drop_ratio": 0.10}, "trials": ROBUSTNESS_TRIALS},
    {"name": "contiguous_subcarrier_mask_30", "fn": perturb_contiguous_subcarrier_mask, "kwargs": {"drop_ratio": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "temporal_mask_10", "fn": perturb_temporal_mask, "kwargs": {"drop_ratio": 0.10}, "trials": ROBUSTNESS_TRIALS},
    {"name": "temporal_mask_30", "fn": perturb_temporal_mask, "kwargs": {"drop_ratio": 0.30}, "trials": ROBUSTNESS_TRIALS},
    {"name": "time_shift_50", "fn": perturb_time_shift, "kwargs": {"shift": 50}, "trials": 1},
    {"name": "crop_resize_75", "fn": perturb_crop_resize, "kwargs": {"keep_ratio": 0.75}, "trials": ROBUSTNESS_TRIALS},
    {"name": "combined_harsh", "fn": perturb_combined_harsh, "kwargs": {}, "trials": ROBUSTNESS_TRIALS},
]


@torch.no_grad()
def evaluate_under_condition(model_obj, loader, condition, trial_seed=42):
    set_seed(trial_seed)

    model_obj.eval()

    preds_all = []
    labels_all = []

    fn = condition["fn"]
    kwargs = condition["kwargs"]

    for x, y in tqdm(loader, desc=condition["name"], leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        x = fn(x, **kwargs)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model_obj(x)

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

        del x, y, logits, preds

    return {
        "accuracy": accuracy_score(labels_all, preds_all),
        "macro_f1": f1_score(labels_all, preds_all, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels_all, preds_all, average="weighted", zero_division=0),
    }


def run_robustness_for_model(model_obj, variant_name):
    rows = []

    for condition in ROBUSTNESS_CONDITIONS:
        trial_results = []

        for t in range(condition["trials"]):
            trial_seed = SEED + 1000 * t

            metrics = evaluate_under_condition(
                model_obj=model_obj,
                loader=test_loader,
                condition=condition,
                trial_seed=trial_seed,
            )

            trial_results.append(metrics)

        accs = np.array([r["accuracy"] for r in trial_results])
        macros = np.array([r["macro_f1"] for r in trial_results])
        weighteds = np.array([r["weighted_f1"] for r in trial_results])

        rows.append({
            "variant": variant_name,
            "condition": condition["name"],
            "trials": condition["trials"],

            "acc_mean": float(accs.mean()),
            "acc_std": float(accs.std(ddof=1)) if len(accs) > 1 else 0.0,

            "macro_f1_mean": float(macros.mean()),
            "macro_f1_std": float(macros.std(ddof=1)) if len(macros) > 1 else 0.0,

            "weighted_f1_mean": float(weighteds.mean()),
            "weighted_f1_std": float(weighteds.std(ddof=1)) if len(weighteds) > 1 else 0.0,
        })

    robustness_df = pd.DataFrame(rows)

    clean_weighted = robustness_df[
        robustness_df["condition"] == "clean"
    ]["weighted_f1_mean"].iloc[0]

    clean_acc = robustness_df[
        robustness_df["condition"] == "clean"
    ]["acc_mean"].iloc[0]

    clean_macro = robustness_df[
        robustness_df["condition"] == "clean"
    ]["macro_f1_mean"].iloc[0]

    robustness_df["acc_drop"] = clean_acc - robustness_df["acc_mean"]
    robustness_df["macro_f1_drop"] = clean_macro - robustness_df["macro_f1_mean"]
    robustness_df["weighted_f1_drop"] = clean_weighted - robustness_df["weighted_f1_mean"]

    return robustness_df

# ============================================================
# RUN ONE VARIANT
# ============================================================

def run_single_ablation_variant(variant_name):
    print("\n" + "=" * 100)
    print(f"RUNNING ABLATION VARIANT: {variant_name}")
    print("=" * 100)

    set_seed(SEED)
    cleanup()

    variant_dir = os.path.join(ABLATION_OUT_DIR, variant_name)
    os.makedirs(variant_dir, exist_ok=True)

    ckpt_path = os.path.join(variant_dir, f"{variant_name}_best.pth")
    history_path = os.path.join(variant_dir, f"{variant_name}_history.csv")
    robustness_path = os.path.join(variant_dir, f"{variant_name}_robustness.csv")
    final_json_path = os.path.join(variant_dir, f"{variant_name}_summary.json")
    report_path = os.path.join(variant_dir, f"{variant_name}_test_report.txt")
    cm_path = os.path.join(variant_dir, f"{variant_name}_confusion_matrix.csv")

    model_obj = DRFTAblationLSTM(
        variant=variant_name,
        num_classes=NUM_CLASSES,
        csi_channels=CSI_CHANNELS,
        num_subcarriers=TARGET_SUBCARRIERS,
        fine_hidden=FINE_HIDDEN,
        coarse_hidden=COARSE_HIDDEN,
        freq_hidden=FREQ_HIDDEN,
        fusion_dim=FUSION_DIM,
        dropout=DROPOUT,
    ).to(DEVICE)

    params = sum(p.numel() for p in model_obj.parameters())
    print(f"Params: {params:,}")

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    optimizer = optim.AdamW(
        model_obj.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=ABLATION_EPOCHS,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda"),
    )

    best_val_macro_f1 = 0.0
    best_val_acc = 0.0
    best_val_weighted_f1 = 0.0
    best_epoch = 0
    patience_counter = 0
    history = []

    for epoch in range(1, ABLATION_EPOCHS + 1):
        cleanup()

        train_metrics = train_one_epoch_model(
            model_obj=model_obj,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            variant_name=variant_name,
        )

        val_metrics = evaluate_model_loader(
            model_obj=model_obj,
            loader=val_loader,
            criterion=criterion,
            split_name=f"{variant_name} Epoch {epoch} Val",
        )

        scheduler.step()

        row = {
            "variant": variant_name,
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
        }

        history.append(row)

        print(
            f"{variant_name} | Epoch [{epoch:02d}/{ABLATION_EPOCHS}] | "
            f"Train Acc: {row['train_acc']*100:.2f}% | "
            f"Train Macro-F1: {row['train_macro_f1']*100:.2f}% | "
            f"Val Acc: {row['val_acc']*100:.2f}% | "
            f"Val Macro-F1: {row['val_macro_f1']*100:.2f}% | "
            f"Val Weighted-F1: {row['val_weighted_f1']*100:.2f}%"
        )

        improved = row["val_macro_f1"] > best_val_macro_f1 + MIN_DELTA

        if improved:
            best_val_macro_f1 = row["val_macro_f1"]
            best_val_acc = row["val_acc"]
            best_val_weighted_f1 = row["val_weighted_f1"]
            best_epoch = epoch
            patience_counter = 0

            torch.save(
                {
                    "model_state_dict": model_obj.state_dict(),
                    "variant": variant_name,
                    "seed": SEED,
                    "best_epoch": best_epoch,
                    "best_val_acc": best_val_acc,
                    "best_val_macro_f1": best_val_macro_f1,
                    "best_val_weighted_f1": best_val_weighted_f1,
                    "config": {
                        "target_subcarriers": TARGET_SUBCARRIERS,
                        "target_time_len": TARGET_TIME_LEN,
                        "csi_channels": CSI_CHANNELS,
                        "num_classes": NUM_CLASSES,
                        "fine_hidden": FINE_HIDDEN,
                        "coarse_hidden": COARSE_HIDDEN,
                        "freq_hidden": FREQ_HIDDEN,
                        "fusion_dim": FUSION_DIM,
                        "robust_training": ROBUST_TRAINING,
                        "aug_prob": AUG_PROB,
                    },
                    "history": history,
                },
                ckpt_path,
            )

            print(
                f"Saved best {variant_name} | "
                f"Epoch {best_epoch} | "
                f"Val Acc {best_val_acc*100:.2f}% | "
                f"Val Macro-F1 {best_val_macro_f1*100:.2f}%"
            )

        else:
            patience_counter += 1
            print(f"No improvement. Patience: {patience_counter}/{ABLATION_PATIENCE}")

        if patience_counter >= ABLATION_PATIENCE:
            print(f"Early stopping {variant_name} at epoch {epoch}. Best epoch: {best_epoch}")
            break

    pd.DataFrame(history).to_csv(history_path, index=False)

    # Load best
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model_obj.load_state_dict(ckpt["model_state_dict"])
    model_obj.to(DEVICE)
    model_obj.eval()

    val_final = evaluate_model_loader(
        model_obj=model_obj,
        loader=val_loader,
        criterion=criterion,
        split_name=f"{variant_name} Final Val",
    )

    test_final = evaluate_model_loader(
        model_obj=model_obj,
        loader=test_loader,
        criterion=criterion,
        split_name=f"{variant_name} Final Test",
    )

    # Reports
    if "INV_LABEL_MAPPING" in globals():
        target_names = [INV_LABEL_MAPPING[i] for i in range(NUM_CLASSES)]
    else:
        target_names = [str(i) for i in range(NUM_CLASSES)]

    test_report = classification_report(
        test_final["labels"],
        test_final["preds"],
        target_names=target_names,
        digits=4,
        zero_division=0,
    )

    test_cm = confusion_matrix(
        test_final["labels"],
        test_final["preds"],
    )

    with open(report_path, "w") as f:
        f.write(test_report)

    pd.DataFrame(test_cm, index=target_names, columns=target_names).to_csv(cm_path)

    # Edge profile
    edge_input_shape = (1, CSI_CHANNELS, TARGET_SUBCARRIERS, TARGET_TIME_LEN)

    size_mb = model_size_mb(model_obj, variant_dir)

    cuda_latency_ms, peak_mem_mb = profile_latency(
        model_obj,
        DEVICE,
        edge_input_shape,
        warmup=20,
        runs=50,
    )

    model_cpu = copy.deepcopy(model_obj).cpu().eval()

    cpu_latency_ms, _ = profile_latency(
        model_cpu,
        torch.device("cpu"),
        edge_input_shape,
        warmup=10,
        runs=30,
    )

    # Robustness
    print("\n" + "-" * 80)
    print(f"Robustness evaluation: {variant_name}")
    print("-" * 80)

    robustness_df = run_robustness_for_model(model_obj, variant_name)

    percent_robustness_df = robustness_df.copy()

    percent_cols = [
        "acc_mean", "acc_std",
        "macro_f1_mean", "macro_f1_std",
        "weighted_f1_mean", "weighted_f1_std",
        "acc_drop", "macro_f1_drop", "weighted_f1_drop",
    ]

    for c in percent_cols:
        percent_robustness_df[c] = percent_robustness_df[c] * 100

    percent_robustness_df.to_csv(robustness_path, index=False)

    # Summary
    summary = {
        "variant": variant_name,
        "seed": SEED,
        "best_epoch": int(best_epoch),

        "params": int(params),
        "model_size_mb": float(size_mb),
        "cuda_latency_ms": float(cuda_latency_ms),
        "cpu_latency_ms": float(cpu_latency_ms),
        "peak_mem_mb": float(peak_mem_mb) if peak_mem_mb is not None else None,

        "val_acc": float(val_final["accuracy"]),
        "val_macro_f1": float(val_final["macro_f1"]),
        "val_weighted_f1": float(val_final["weighted_f1"]),

        "test_acc": float(test_final["accuracy"]),
        "test_macro_f1": float(test_final["macro_f1"]),
        "test_weighted_f1": float(test_final["weighted_f1"]),

        "ckpt_path": ckpt_path,
        "history_path": history_path,
        "robustness_path": robustness_path,
        "report_path": report_path,
        "confusion_matrix_path": cm_path,
    }

    with open(final_json_path, "w") as f:
        json.dump(summary, f, indent=4)

    print("\n" + "=" * 80)
    print(f"{variant_name} FINAL SUMMARY")
    print("=" * 80)
    print(json.dumps(summary, indent=4))

    print("\nRobustness results:")
    display(percent_robustness_df)

    del model_obj, model_cpu
    cleanup()

    return summary, percent_robustness_df

# ============================================================
# RUN ALL ABLATIONS
# ============================================================

all_summaries = []
all_robustness = []

for variant_name in RUN_VARIANTS:
    summary, robustness_df = run_single_ablation_variant(variant_name)
    all_summaries.append(summary)
    all_robustness.append(robustness_df)

summary_df = pd.DataFrame(all_summaries)
robustness_all_df = pd.concat(all_robustness, ignore_index=True)

summary_path = os.path.join(
    ABLATION_OUT_DIR,
    "drft_lstm_ablation_d1_d4_summary.csv",
)

robustness_all_path = os.path.join(
    ABLATION_OUT_DIR,
    "drft_lstm_ablation_d1_d4_robustness_all.csv",
)

summary_df.to_csv(summary_path, index=False)
robustness_all_df.to_csv(robustness_all_path, index=False)

# ============================================================
# COMPACT PAPER TABLES
# ============================================================

key_conditions = [
    "clean",
    "gaussian_noise_0.20",
    "random_subcarrier_mask_30",
    "random_subcarrier_mask_50",
    "contiguous_subcarrier_mask_30",
    "temporal_mask_30",
    "crop_resize_75",
    "combined_harsh",
]

compact_rows = []

for variant_name in RUN_VARIANTS:
    srow = summary_df[summary_df["variant"] == variant_name].iloc[0].to_dict()
    rsub = robustness_all_df[robustness_all_df["variant"] == variant_name]

    row = {
        "variant": variant_name,
        "params": srow["params"],
        "model_size_mb": srow["model_size_mb"],
        "cuda_latency_ms": srow["cuda_latency_ms"],
        "cpu_latency_ms": srow["cpu_latency_ms"],
        "test_acc": srow["test_acc"] * 100,
        "test_macro_f1": srow["test_macro_f1"] * 100,
        "test_weighted_f1": srow["test_weighted_f1"] * 100,
    }

    for cond in key_conditions:
        cdf = rsub[rsub["condition"] == cond]

        if len(cdf) > 0:
            row[f"{cond}_weighted_f1"] = float(cdf.iloc[0]["weighted_f1_mean"])
            row[f"{cond}_drop"] = float(cdf.iloc[0]["weighted_f1_drop"])
        else:
            row[f"{cond}_weighted_f1"] = np.nan
            row[f"{cond}_drop"] = np.nan

    compact_rows.append(row)

compact_df = pd.DataFrame(compact_rows)

compact_path = os.path.join(
    ABLATION_OUT_DIR,
    "drft_lstm_ablation_d1_d4_compact_paper_table.csv",
)

compact_df.to_csv(compact_path, index=False)

print("\n" + "=" * 100)
print("DRFT-LSTM ABLATION SUMMARY")
print("=" * 100)
display(summary_df)

print("\n" + "=" * 100)
print("DRFT-LSTM ABLATION COMPACT PAPER TABLE")
print("=" * 100)
display(compact_df)

# ============================================================
# ZIP RESULTS
# ============================================================

zip_path = "/kaggle/working/drft_lstm_ablation_d1_d4_results.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=ABLATION_OUT_DIR,
)

print("\nSaved:")
print(summary_path)
print(robustness_all_path)
print(compact_path)

print("\nDownload:")
display(FileLink(zip_path))

cleanup()